### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [1]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Load environment variables
load_dotenv()

True

### Data Ingestion And Processing


In [2]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [4]:
## text splitting
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n", " "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(f"Number of chunks created: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:\n{chunk.page_content}\nMetadata: {chunk.metadata}")


Number of chunks created: 4

Chunk 1:
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}

Chunk 2:
Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
Metadata: {'source': 'ML Basics', 'page': 1, 'topic': 'ML'}

Chunk 3:
Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
Metadata: {'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}

Chunk 4:
Natural L

In [5]:
### load the embedding models
import os
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [6]:
# Initialize OpenAI embeddings with the latest model

embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

## Example: create a embedding for a single text
sample_text="What is machine learning"
sample_embedding=embeddings.embed_query(sample_text)
sample_embedding

[-0.0059221177361905575,
 -0.005889697000384331,
 0.0005751235294155777,
 -0.033544257283210754,
 0.021192103624343872,
 0.02213229238986969,
 -0.0007868689717724919,
 0.00929923728108406,
 -0.022564563900232315,
 0.037823744118213654,
 0.015183531679213047,
 -0.03676467761397362,
 -0.03397652879357338,
 0.0016048074467107654,
 0.026735983788967133,
 0.016015654429793358,
 0.0008530605118721724,
 -0.030864175409078598,
 0.023385880514979362,
 0.004903578199446201,
 3.242035018047318e-05,
 0.032550033181905746,
 0.042276136577129364,
 -0.029307996854186058,
 0.015961619094014168,
 -0.02092193439602852,
 0.0266279149800539,
 0.01239538099616766,
 0.00744047062471509,
 -0.006019378546625376,
 -0.009423515759408474,
 -0.029027020558714867,
 -0.03127483278512955,
 0.02755729854106903,
 0.04173579812049866,
 0.00043159592314623296,
 -0.0080564571544528,
 -0.010633875615894794,
 -0.055503640323877335,
 0.0025855230633169413,
 -0.05632495880126953,
 -0.016264209523797035,
 0.03028060868382454,

In [11]:
texts=["AI","MAchine learning","Deep Learning","Neural Network"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.008167067542672157, -0.024646570906043053, 0.002890851115807891, 0.0251751821488142, 0.006528367754071951, -0.02828078344464302, -0.005031732842326164, 0.020985927432775497, -0.03687074035406113, 0.012865113094449043, -0.0030659539625048637, -0.020153362303972244, 0.000287433183984831, -0.03274755924940109, 0.006425948813557625, -0.025307336822152138, -0.031055999919772148, -0.054394252598285675, 0.03277399018406868, -0.018369292840361595, 0.01663808710873127, 0.04831520840525627, -0.024937307462096214, 0.014351836405694485, 0.029364438727498055, 0.0040901415050029755, 0.009323407895863056, 0.01335407979786396, 0.0024894357193261385, -0.022584980353713036, 0.032113224267959595, -0.028016475960612297, 0.005368723534047604, -0.0381922721862793, -0.01670416258275509, 0.014365051873028278, -0.038615163415670395, -0.010387240909039974, -0.010532609187066555, -0.019188642501831055, 0.03203393518924713, 0.014682219363749027, -0.021514538675546646, 0.016083043068647385, -0.0118144955486059

In [12]:
print(batch_embeddings[1])

[-0.0182332843542099, 0.010754693299531937, 0.01752162165939808, -0.03528864309191704, 0.03744817152619362, 0.0260861124843359, 0.015914246439933777, 0.008398844860494137, -0.010932608507573605, 0.03489600494503975, -0.006607418414205313, -0.06866316497325897, -0.03239291533827782, 0.008987806737422943, 0.03658926859498024, 0.01869954541325569, 0.01629461720585823, -0.01539890468120575, 0.020625943318009377, 0.012993975542485714, -0.0026334580034017563, 0.03779173269867897, 0.03408617898821831, -0.030012525618076324, 0.020675022155046463, 0.007920312695205212, 0.03612300753593445, 0.0376935712993145, 0.010539967566728592, -0.03484692424535751, -0.010883528739213943, -0.024221066385507584, -0.01395103894174099, -0.013840609230101109, 0.01196329202502966, -0.001851242734119296, -0.02763213776051998, 0.027828458696603775, -0.045080140233039856, -0.00519022811204195, -0.01974250003695488, -0.04277336969971657, 0.034356120973825455, 0.06827051937580109, -0.020589131861925125, -0.03484692424

In [13]:
### Compare Embedding using cosine similarity

def compare_embeddings(text1:str,text2:str):
    """Compare semantic simialrity of 2 texts usign embeddings"""

    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    ## Calculate the simialrity score

    similarity=np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [14]:
# Test semantic similarity
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")


Semantic Similarity Examples:
'AI' vs 'Artificial Intelligence': 0.563


In [19]:
print(f"'AI' vs 'Pizza': {compare_embeddings('AI', 'Pizza'):.3f}")

'AI' vs 'Pizza': 0.254


In [17]:
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML in Data Science'):.3f}")

'Machine Learning' vs 'ML': 0.617


### Create FAISS Vector Store

In [20]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [21]:
vectorstore

In [22]:
## Save vector tore for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [23]:
## load vector store
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [24]:
## Similarity Search 
query="What is deep learning"

results=vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='65145d3d-0b83-4462-97b2-7c2baa957f10', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='460c3126-c88b-4c85-93ff-439596bf3ab4', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='fd10115e-ad43-439b-b36c-a6f78556bcd5', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [25]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [27]:
### Similarity Search with score
results_with_scores=vectorstore.similarity_search_with_score(query,k=3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score: 0.556
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 1.208
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Score: 1.274
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [28]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [29]:
### Search with metadata filtering
filter_dict={"topic":"ML"}
filtered_results=vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='460c3126-c88b-4c85-93ff-439596bf3ab4', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [30]:
len(filtered_results)

1

### Build RAG Chain With LCEL 

In [36]:
## LLM GROQ LLM
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model(model="groq:groq/compound")
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x00000126B25A2850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000126A57D3110>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [37]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 232, 'total_tokens': 267, 'completion_time': 0.084772, 'completion_tokens_details': None, 'prompt_time': 0.007426, 'prompt_tokens_details': None, 'queue_time': 0.10966, 'total_time': 0.092198}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--67203ce2-0a41-42e1-ac4e-e3cb4b65c868-0', usage_metadata={'input_tokens': 232, 'output_tokens': 35, 'total_tokens': 267})

In [38]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [39]:
## Basic retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [40]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000126A5C0E510>, search_kwargs={'k': 3})

In [41]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [44]:
simple_rag_chain=(
    {
        "context":retriever | format_docs,
        "question":RunnablePassthrough() 
    }
    | simple_prompt
    | llm
    |StrOutputParser()

)

In [45]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000126A5C0E510>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x00000126B25A2850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000126A57D3110>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [54]:
simple_rag_chain.invoke("What are the different types of machine learning?")

'The different types of machine learning are:\n\n- **Supervised learning**  \n- **Unsupervised learning**  \n- **Reinforcement learning**'

In [46]:
### Conversational RAg Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [47]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [76]:
result = conversational_rag.invoke(
    {
        "input": "what is ML?",
        "chat_history": []
    }
)

In [78]:
chat_history = ([
                    HumanMessage(content="what is ML?"),
                    AIMessage(content=result)
                ])
chat_history

[HumanMessage(content='what is ML?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine Learning (ML) is a branch of artificial intelligence that lets computers **learn from data instead of being explicitly programmed**. By analyzing data, ML algorithms discover patterns and build models that can make predictions or decisions on new, unseen inputs. It encompasses approaches such as supervised, unsupervised, and reinforcement learning.', additional_kwargs={}, response_metadata={})]

In [79]:
result = conversational_rag.invoke({
            "input": "explain more about it's types.",
            "chat_history": chat_history
        })

In [80]:
result

'**Machine‑Learning (ML) – a quick recap**\n\nFrom the supplied documents we know that:\n\n* **ML is a subset of Artificial Intelligence (AI)** – it lets systems learn from data instead of being hand‑coded for every task.  \n* The **basic taxonomy** mentioned in *Document\u202f2* lists three “common types”: **supervised, unsupervised, and reinforcement learning**.\n\nBelow is a fuller description of those three core types **plus** two increasingly popular variants (semi‑supervised and self‑supervised) that many modern resources now treat as distinct categories.\n\n---\n\n## 1. Supervised Learning\n| Aspect | Details |\n|--------|---------|\n| **What it is** | The algorithm is trained on **labeled data** – each training example comes with the “correct answer” (the target or label). |\n| **Goal** | Learn a mapping from inputs → outputs so the model can predict the label for new, unseen inputs. |\n| **Typical problems** | • **Classification** – assign a discrete class (e.g., spam vs.\u202

In [65]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")
streaming_rag_chain

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000126A5C0E510>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x00000126B25A2850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000126A57D3110>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [68]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [69]:
test_rag_chains("What is the difference between AI and machine learning")

Question: What is the difference between AI and machine learning

1. Simple RAG Chain:
Answer: **Reasoning from the provided documents**

1. **Document 1 (ML Basics)** – States that *“Machine Learning is a subset of AI that enables systems to learn from data.”*  
   - This tells us that ML belongs inside the broader AI field and is defined by its data‑driven learning approach.

2. **Document 2 (AI Introduction)** – Defines AI as *“the simulation of human intelligence in machines.”*  
   - AI covers any technique that makes a machine behave intelligently (reasoning, problem‑solving, perception, etc.), whether it learns from data or is explicitly programmed.

3. **Document 3 (Deep Learning)** – Notes that *“Deep Learning is a subset of machine learning.”*  
   - This reinforces the hierarchy: Deep Learning ⊂ Machine Learning ⊂ AI.

**Answer to the question**

The difference between AI and machine learning is one of scope:

- **Artificial Intelligence (AI)** is the broad discipline concer

In [70]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: **Reasoning based on the provided context**

1. **Definition of Artificial Intelligence (AI)** – from Document 2 (AI Introduction):
   - AI is the simulation of human intelligence in machines.
   - These systems are designed to think like humans and mimic their actions.
   - AI can be categorized into narrow AI and general AI.

2. **Definition of Machine Learning (ML)** – from Document 1 (ML Basics):
   - Machine Learning is a *subset of AI* that enables systems to learn from data.
   - Instead of being explicitly programmed, ML algorithms find patterns in data.
   - Common types include supervised, unsupervised, and reinforcement learning.

3. **Relationship** – The first sentence of Document 1 explicitly states that ML is a subset of AI, meaning all machine‑learning systems are AI systems, but not all AI systems are machine‑learning systems.

4. **Key distinction** –  
   - **AI** is the

In [71]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")


3. Conversational RAG Example:
Q1: What is machine learning?
A1: Machine learning (ML) is a branch of artificial intelligence that enables systems to learn from data rather than being explicitly programmed. ML algorithms automatically discover patterns and relationships within data, allowing them to make predictions or decisions. Common approaches include supervised learning, unsupervised learning, and reinforcement learning.


In [72]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [75]:
# Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: **Reasoning (based on the supplied documents)**  

1. **Identify what the documents say about machine learning (ML) and deep learning (DL):**  
   - *Document 2 (ML Basics)* tells us that **Machine Learning is a subset of AI that enables systems to learn from data**. Instead of being explicitly programmed, **ML algorithms find patterns in data**. It also lists the common types of ML (supervised, unsupervised, reinforcement learning).  
   - *Document 1 (Deep Learning)* adds that **Deep Learning is a subset of ML that uses artificial neural networks with many layers** to extract higher‑level features from raw input. DL has driven breakthroughs in computer vision, NLP, and speech recognition.  

2. **Identify what the documents say about NLP (relevant because it shows how ML/DL are applied):**  
   - *Document 3 (NLP Overview)* explains that **Natural Language Processing combines computational linguistics with ML and DL models** 